In [6]:
from pathlib import Path
import sys

# Locate the simulator module.
candidates = (Path.cwd(), Path.cwd().parent)
project_dir = next(
    (path for path in candidates
     if (path / "mini8.py").is_file() and (path / "notebook_tools.py").is_file()),
    None,
)
if project_dir is None:
    raise RuntimeError("Open the project containing mini8.py and restart the kernel.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from notebook_tools import CPU
from mini8 import assemble, disassemble

print("MINI-8 is ready.")

MINI-8 is ready.


SETUP

In [7]:
source = """
; Write your MINI-8 instructions here.
HALT
"""
practice_cpu = CPU(source, title="Lecture 2 practice")
practice_cpu.show()

TASK 1 - FOLLOW THE STORED FLAG AND THE PROGRAM COUNTER
----------------------------------------------------------------------
Notebook reference: Sections 1-4.2 and 8.

a) For each pair below, predict the flag after CMP R0, R1. Then state
   whether each of JE, JNE, JL, and JGE would take its jump immediately
   afterwards. Present your answers as a table.

       R0     R1
       2      8
       8      8
       255    1

   Explain why the last comparison is unsigned. Does CMP change either
   register? Does a conditional jump perform another comparison?

b) Trace this program on paper before running it:


In [15]:
source = """
LDI R0, 6
LDI R1, 4
CMP R0, R1
SUB R0, R1
JL small
LDI R2, 9
JMP finish
small:
    LDI R2, 1
finish:
    OUT R2
    HALT
"""
cpu = CPU(source, title="Lecture 2 practice")
cpu.run()

Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 6",0 → 1,6,0,0,0,=,—,—
2,"LDI R1, 4",1 → 2,6,4,0,0,=,—,—
3,"CMP R0, R1",2 → 3,6,4,0,0,>,—,—
4,"SUB R0, R1",3 → 4,2,4,0,0,>,—,—
5,JL 7,4 → 5 · no jump,2,4,0,0,>,—,—
6,"LDI R2, 9",5 → 6,2,4,9,0,>,—,—
7,JMP 8,6 → 8 ↶ jump,2,4,9,0,>,—,—
8,OUT R2,8 → 9,2,4,9,0,>,—,9
9,HALT,9 → 9 · HALT,2,4,9,0,>,—,—


c) Check your trace using step(). Pay particular attention to the flag
   after SUB. Then insert CMP R0, R1 immediately before JL small.
   Predict how the branch and output change, then run a fresh CPU to check.

In [16]:
source = """
LDI R0, 6
LDI R1, 4
CMP R0, R1
SUB R0, R1
CMP R0, R1
JL small
LDI R2, 9
JMP finish
small:
    LDI R2, 1
finish:
    OUT R2
    HALT
"""
cpu = CPU(source, title="Lecture 2 practice")
cpu.run()

Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 6",0 → 1,6,0,0,0,=,—,—
2,"LDI R1, 4",1 → 2,6,4,0,0,=,—,—
3,"CMP R0, R1",2 → 3,6,4,0,0,>,—,—
4,"SUB R0, R1",3 → 4,2,4,0,0,>,—,—
5,"CMP R0, R1",4 → 5,2,4,0,0,<,—,—
6,JL 8,5 → 8 ↶ jump,2,4,0,0,<,—,—
7,"LDI R2, 1",8 → 9,2,4,1,0,<,—,—
8,OUT R2,9 → 10,2,4,1,0,<,—,1
9,HALT,10 → 10 · HALT,2,4,1,0,<,—,—


TASK 2 - TRANSLATE AN IF/ELSE AND TEST ITS BOUNDARY
----------------------------------------------------------------------
Notebook reference: Section 5.

Translate this source-level rule into a complete MINI-8 program:

    if x < 12:
        y = x + 3
    else:
        y = x - 2
    print(y)

Keep x in R0 and the final y in R2. Load the input at the beginning and
leave R0 unchanged. Use the remaining registers for the boundary and
constants. Use labels for the two paths and their shared continuation.
Output y exactly once and finish with HALT. Do not load a precomputed y.


a) Before coding, draw the two paths or describe them in plain English.
   Identify the comparison and the condition for entering each path.

In [17]:
source = """
LDI R0, x
;LDI R1, y
CMP R0, 12
JL less
JGE great
JMP finish

less:
MOV R2, R0
ADD R2, 3

great:
MOV R2, R0
SUB R2, 2

finish:
OUT R2
HALT

"""
cpu = CPU(source, title="Lecture 2 practice")
cpu.run()

Mini8Error: Line 2: Invalid number or unknown symbol: 'x'